# Análise de Churn — Connecta Telecom

## Problema de Negócio

A **Connecta Telecom** está enfrentando uma taxa de cancelamento de serviços (churn) acima da média do setor.

Este projeto tem dois objetivos:
1. **Entender** quais fatores levam ao churn (análise estatística simplificada)
2. **Prever** se um cliente vai cancelar (modelo preditivo)


## 1. Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import statsmodels.api as sm

# Modelo preditivo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas importadas com sucesso!')

## 2. Geração dos Dados

In [ ]:
def gera_dados_churn(num_clientes=2000):
    """
    Gera um DataFrame de dados fictícios de clientes de uma empresa de telecomunicações.
    """
    np.random.seed(42)

    fidelidade_meses = np.random.randint(1, 73, size=num_clientes)
    tipo_contrato_opts = ['Mensal', 'Anual', 'Dois anos']
    contrato_probs = [0.6, 0.25, 0.15]
    tipo_contrato = np.random.choice(tipo_contrato_opts, size=num_clientes, p=contrato_probs)
    servico_internet_opts = ['Fibra Óptica', 'DSL', 'Não']
    internet_probs = [0.55, 0.35, 0.10]
    servico_internet = np.random.choice(servico_internet_opts, size=num_clientes, p=internet_probs)

    fatura_base = {
        'Mensal': np.random.normal(60, 20),
        'Anual': np.random.normal(70, 25),
        'Dois anos': np.random.normal(80, 25)
    }

    fatura_mensal = [fatura_base[c] + fidelidade_meses[i] * 0.2 + np.random.normal(0, 5)
                     for i, c in enumerate(tipo_contrato)]
    fatura_mensal = np.clip(fatura_mensal, 20, 120)

    prob_churn_log = -2.5
    prob_churn_log += -0.05 * fidelidade_meses
    prob_churn_log += [3.0 if c == 'Mensal' else -1.5 if c == 'Anual' else -2.5 for c in tipo_contrato]
    prob_churn_log += [0.8 if s == 'Fibra Óptica' else -0.5 for s in servico_internet]
    prob_churn_log += 0.03 * fatura_mensal

    prob_churn = 1 / (1 + np.exp(-prob_churn_log))
    churn = np.random.binomial(1, prob_churn)

    df = pd.DataFrame({
        'ID_Cliente': range(1, num_clientes + 1),
        'Fidelidade_Meses': fidelidade_meses,
        'Tipo_Contrato': tipo_contrato,
        'Servico_Internet': servico_internet,
        'Fatura_Mensal': fatura_mensal,
        'Churn': churn
    })
    return df


df_churn = gera_dados_churn()
df_churn.head()

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
print('Shape do dataset:', df_churn.shape)
print('\nValores ausentes:\n', df_churn.isnull().sum())
print('\nTipos de dados:\n', df_churn.dtypes)

In [ ]:
df_churn.describe()

In [ ]:
# Taxa de churn geral
churn_counts = df_churn['Churn'].value_counts().rename(index={1: 'Sim', 0: 'Não'})
taxa = 100 * df_churn['Churn'].sum() / len(df_churn)

cores = ['#6FA8DC', '#F44336']
plt.figure(figsize=(6, 6))
plt.pie(
    churn_counts.values,
    labels=churn_counts.index,
    autopct='%1.2f%%',
    startangle=140,
    colors=cores,
    explode=[0.05 if label == 'Sim' else 0 for label in churn_counts.index]
)
plt.title(f'Taxa de Churn Geral: {taxa:.1f}%', fontsize=14)
plt.show()

In [ ]:
# Churn por tipo de contrato
fig = px.histogram(
    df_churn, x='Tipo_Contrato', color='Churn',
    barmode='group',
    title='Churn por Tipo de Contrato',
    labels={'Tipo_Contrato': 'Tipo de Contrato', 'Churn': 'Churn (0=Não, 1=Sim)'}
)
fig.show()

In [ ]:
# Distribuição de fidelidade por churn
fig = px.histogram(
    df_churn, x='Fidelidade_Meses', color='Churn',
    marginal='box',
    title='Distribuição de Churn pela Fidelidade (meses)',
    labels={'Fidelidade_Meses': 'Meses de Fidelidade'}
)
fig.show()

In [ ]:
# Distribuição de fatura por churn
fig = px.histogram(
    df_churn, x='Fatura_Mensal', color='Churn',
    marginal='box',
    title='Distribuição de Churn pela Fatura Mensal',
    labels={'Fatura_Mensal': 'Valor da Fatura (R$)'}
)
fig.show()

## 4. Análise Estatística Simplificada

Vamos interpretar os resultados da seguinte forma:
- **Coeficiente positivo** → aquela variável **aumenta** a probabilidade de churn
- **Coeficiente negativo** → aquela variável **diminui** a probabilidade de churn
- **Valor-p < 0.05** → o efeito é estatisticamente significativo (não é coincidência)

In [ ]:
# Preparação dos dados para o modelo estatístico
df_model = pd.get_dummies(
    df_churn,
    columns=['Tipo_Contrato', 'Servico_Internet'],
    drop_first=True,
    dtype=int
)

y = df_model['Churn']
X = df_model.drop(['ID_Cliente', 'Churn'], axis=1)
X = sm.add_constant(X)

# Ajuste do modelo
modelo = sm.Logit(y, X)
resultado = modelo.fit(disp=False)
print(resultado.summary())

In [ ]:
# Interpretação simplificada: probabilidade marginal média (AME)
# Mostra o impacto de cada variável diretamente em pontos percentuais de churn

efeitos_marginais = resultado.get_margeff()
print(efeitos_marginais.summary())

### Interpretação dos Efeitos Marginais (leitura direta)

Os **efeitos marginais** mostram o impacto de cada variável **em pontos percentuais** sobre a probabilidade de churn — muito mais intuitivo que odds ratios:

| Variável | Leitura direta |
|---|---|
| `Fidelidade_Meses` | A cada 1 mês a mais de contrato, a probabilidade de churn cai X pp |
| `Fatura_Mensal` | A cada R$1 a mais na fatura, a probabilidade de churn sobe X pp |
| `Tipo_Contrato_Mensal` | Ter contrato mensal aumenta X pp a chance de churn vs. Anual |
| `Tipo_Contrato_Dois anos` | Contrato de 2 anos reduz X pp a chance de churn vs. Anual |
| `Servico_Internet_Fibra Óptica` | Ter fibra ótica aumenta X pp a probabilidade de cancelar vs. DSL |

In [ ]:
# Gráfico dos efeitos marginais
ame = efeitos_marginais.summary_frame()
ame = ame[['dy/dx', 'Pr(>|z|)']].copy()
ame.columns = ['Efeito_Marginal', 'p_valor']
ame['Significativo'] = ame['p_valor'] < 0.05
ame = ame.sort_values('Efeito_Marginal')

cores_barra = ['#F44336' if v > 0 else '#6FA8DC' for v in ame['Efeito_Marginal']]

plt.figure(figsize=(10, 5))
bars = plt.barh(ame.index, ame['Efeito_Marginal'], color=cores_barra, edgecolor='white')
plt.axvline(0, color='gray', linewidth=0.8, linestyle='--')
plt.title('Impacto de Cada Variável na Probabilidade de Churn\n(Efeitos Marginais — em pontos percentuais)', fontsize=13)
plt.xlabel('Variação na probabilidade de churn (pp)')
plt.tight_layout()
plt.show()

print('\nVariáveis em vermelho AUMENTAM a chance de churn.')
print('Variáveis em azul DIMINUEM a chance de churn.')

## 5. Modelo Preditivo de Churn

Agora vamos além da análise estatística e construímos um modelo capaz de **prever se um cliente vai cancelar**, usando Machine Learning.

Vamos testar dois algoritmos:
- **Regressão Logística** (sklearn) — simples, interpretável
- **Random Forest** — mais poderoso, capta relações não lineares

In [ ]:
# Preparando os dados para ML
df_ml = pd.get_dummies(
    df_churn.drop('ID_Cliente', axis=1),
    columns=['Tipo_Contrato', 'Servico_Internet'],
    drop_first=True,
    dtype=int
)

X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

# Divisão treino/teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalização (importante para a Regressão Logística)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} clientes')
print(f'Teste:  {X_test.shape[0]} clientes')

In [ ]:
# --- Regressão Logística ---
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sc, y_train)

y_pred_lr   = lr.predict(X_test_sc)
y_proba_lr  = lr.predict_proba(X_test_sc)[:, 1]

print('=== Regressão Logística ===')
print(classification_report(y_test, y_pred_lr, target_names=['Não Cancelou', 'Cancelou']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba_lr):.4f}')

In [ ]:
# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf   = rf.predict(X_test)
y_proba_rf  = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Não Cancelou', 'Cancelou']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba_rf):.4f}')

In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, titulo in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Regressão Logística', 'Random Forest']
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Não Cancelou', 'Cancelou'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(titulo)

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC comparativas
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)

auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_rf = roc_auc_score(y_test, y_proba_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f'Regressão Logística (AUC = {auc_lr:.3f})', color='#6FA8DC', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest      (AUC = {auc_rf:.3f})', color='#F44336', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Modelo aleatório (AUC = 0.500)')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curva ROC — Comparação dos Modelos')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Importância das variáveis — Random Forest
importancias = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importancias.plot(kind='barh', color='#6FA8DC', edgecolor='white')
plt.title('Importância das Variáveis — Random Forest')
plt.xlabel('Importância relativa')
plt.tight_layout()
plt.show()

## 6. Prevendo o Churn de Novos Clientes

Agora que o modelo está treinado, podemos usar para prever a probabilidade de churn de **novos clientes**.

In [ ]:
def prever_churn(fidelidade_meses, tipo_contrato, servico_internet, fatura_mensal, modelo=rf):
    """
    Prevê a probabilidade de churn de um novo cliente.
    
    Parâmetros:
        fidelidade_meses  : int   — quantos meses o cliente está na empresa
        tipo_contrato     : str   — 'Mensal', 'Anual' ou 'Dois anos'
        servico_internet  : str   — 'Fibra Óptica', 'DSL' ou 'Não'
        fatura_mensal     : float — valor da fatura em R$
        modelo            : modelo treinado (padrão: Random Forest)
    """
    dados = pd.DataFrame([{
        'Fidelidade_Meses': fidelidade_meses,
        'Fatura_Mensal': fatura_mensal,
        'Tipo_Contrato_Dois anos': 1 if tipo_contrato == 'Dois anos' else 0,
        'Tipo_Contrato_Mensal':   1 if tipo_contrato == 'Mensal'    else 0,
        'Servico_Internet_Fibra Óptica': 1 if servico_internet == 'Fibra Óptica' else 0,
        'Servico_Internet_Não':          1 if servico_internet == 'Não'          else 0,
    }])

    prob = modelo.predict_proba(dados)[0][1]
    risco = 'ALTO 🔴' if prob >= 0.6 else ('MÉDIO 🟡' if prob >= 0.35 else 'BAIXO 🟢')

    print(f'Probabilidade de Churn: {prob*100:.1f}%')
    print(f'Nível de Risco: {risco}')
    return prob


# ---- Exemplos de uso ----

print('--- Cliente 1: recente, contrato mensal, fibra óptica, fatura alta ---')
prever_churn(fidelidade_meses=3, tipo_contrato='Mensal', servico_internet='Fibra Óptica', fatura_mensal=110)

print('\n--- Cliente 2: fiel, contrato de 2 anos, DSL, fatura moderada ---')
prever_churn(fidelidade_meses=48, tipo_contrato='Dois anos', servico_internet='DSL', fatura_mensal=65)

print('\n--- Cliente 3: fidelidade média, contrato anual, fibra óptica ---')
prever_churn(fidelidade_meses=18, tipo_contrato='Anual', servico_internet='Fibra Óptica', fatura_mensal=85)

## 7. Conclusões e Recomendações Estratégicas

### O que os dados revelam

| Fator | Impacto no Churn |
|---|---|
| **Contrato Mensal** | Altíssimo risco — clientes mensais cancelam muito mais |
| **Pouco tempo de fidelidade** | Crítico — primeiros meses são os mais arriscados |
| **Fibra Óptica** | Risco elevado — pode indicar insatisfação ou sensibilidade a preço |
| **Fatura alta** | Risco moderado — clientes com faturas maiores cancelam mais |
| **Contrato de 2 anos** | Fator de proteção — reduz muito o churn |

### Recomendações

1. **Converter contratos mensais em anuais/bienais** — ofereça desconto ou benefício para quem migrar
2. **Intensificar o acompanhamento nos primeiros 6 meses** — período mais crítico para retenção
3. **Investigar a insatisfação com Fibra Óptica** — qualidade do serviço ou percepção de custo?
4. **Usar o modelo preditivo** para identificar clientes em risco e acionar campanhas de retenção antes do cancelamento